In [1]:
from transformers import AutoConfig, AutoModelForCausalLM
from collections import defaultdict
import re

In [2]:
config = AutoConfig.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_config(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

In [3]:
groups = defaultdict(list)

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue

    pattern = re.sub(r"\.\d+\.", ".N.", name)
    groups[pattern].append(p.shape)

for pattern, shapes in groups.items():
    if shapes:
        print(f"{pattern} x {len(shapes)} (Shape: {shapes[0]})")
    else:
        print(f"{pattern} x {len(shapes)}")

model.embed_tokens.weight x 1 (Shape: torch.Size([151936, 896]))
model.layers.N.self_attn.q_proj.weight x 24 (Shape: torch.Size([896, 896]))
model.layers.N.self_attn.q_proj.bias x 24 (Shape: torch.Size([896]))
model.layers.N.self_attn.k_proj.weight x 24 (Shape: torch.Size([128, 896]))
model.layers.N.self_attn.k_proj.bias x 24 (Shape: torch.Size([128]))
model.layers.N.self_attn.v_proj.weight x 24 (Shape: torch.Size([128, 896]))
model.layers.N.self_attn.v_proj.bias x 24 (Shape: torch.Size([128]))
model.layers.N.self_attn.o_proj.weight x 24 (Shape: torch.Size([896, 896]))
model.layers.N.mlp.gate_proj.weight x 24 (Shape: torch.Size([4864, 896]))
model.layers.N.mlp.up_proj.weight x 24 (Shape: torch.Size([4864, 896]))
model.layers.N.mlp.down_proj.weight x 24 (Shape: torch.Size([896, 4864]))
model.layers.N.input_layernorm.weight x 24 (Shape: torch.Size([896]))
model.layers.N.post_attention_layernorm.weight x 24 (Shape: torch.Size([896]))
model.norm.weight x 1 (Shape: torch.Size([896]))


## Conclusions

1. Qwen2.5-0.5B has 24 layers
2. For the Muon optimizer, `self_attn.q_proj.weight`, `self_attn.k_proj.weight`, `self_attn.v_proj.weight`, `self_attn.o_proj.weight`, `mlp.gate_proj.weight`, `mlp.up_proj.weight`, `mlp.down_proj.weight` may be suitable.